# 3.1 Cross-correlation after prewhitening

Complementary analysis to the main DLNM (`03_2_dlnm.ipynb`). Preregistration: *"lagged cross-correlations are computed after removing seasonality and prewhitening both series."* The methods document (Metodologija) adds: fit a time series model to each exposure, filter the arrivals with the same filter, compute the CCF of the residuals for lags 0 to 21 days with 95% limits ±1.96/√n, check exposure collinearity with a Spearman matrix, and use the lags found here **only to plan the main model**: every final estimate comes from the regression.

**What this notebook does.**

1. Describes the primary outcomes: daily, weekly and yearly level (§2), a decomposition into trend, weekly and yearly seasonality (§3), and their autocorrelation after seasonal adjustment (§4).
2. Computes the prewhitened cross-correlation for H1–H3, and for PM2.5 as a secondary exposure analysed exactly like H3 (PM10), with negative-control outcomes, a dummy exposure and sensitivity analyses (§5–6).
3. Checks collinearity between the exposures (§7).
4. Screens all other exposures against six arrival streams, as an exploratory analysis (§8).
5. Lists, for each hypothesis, the lags whose correlation lies outside the limits, as input to the DLNM lag choice (§9).

**Reading the results.** A CCF value at lag *k* ≥ 0 is the correlation of the prewhitened exposure on day *t* with the prewhitened arrivals on day *t + k* (exposure first). Negative lags (arrivals first) cannot be an effect of the exposure, so correlations there indicate residual confounding or reverse timing. **No hypothesis test is made here**: the Holm-corrected tests for H1–H3 come from the DLNM. Two kinds of limits are shown: pointwise ±1.96/√n, which about 1 lag in 20 will cross by chance, and a simultaneous 95% band from the dummy-exposure simulation, which holds for all lags 0 to *L* together.

**Decisions not fixed in the plan** (to add to `docs/deviations_log.md` where they differ from it):

| Decision | Choice | Reason |
|---|---|---|
| Disease arrivals (H2, H3) | `arr_medical_mts_all` (Enota za bolezni + INP) | as in the codebook: routing between the two units changed in 2021–23 |
| Removing seasonality | OLS of each series on a natural cubic spline of time (1 df per year), 3 annual Fourier harmonics, day of week, holidays, the COVID-19 pandemic period (to 2023-03-31) and the two structural breaks in the panel (`hp_referral_era`, `triage_break_2024_06`). Arrivals on the log scale | a fixed annual shape plus a slow trend keeps each year's own deviations (heat waves, the timing and size of each flu season). The DLNM convention of 7 df per year would absorb most of an influenza wave; it is run as a sensitivity analysis |
| Prewhitening filter | AR(*p*) fitted to the adjusted exposure, *p* chosen by AIC (daily up to 14, weekly up to 6), the same filter applied to the adjusted arrivals | Box–Jenkins prewhitening; an AR model is the usual choice (e.g. `TSA::prewhiten` in R). The plan says "ARIMA"; AR is the special case without MA terms, and differencing is not needed after seasonal adjustment |
| Influenza (H2) | weekly analysis, lags 0–3 weeks | `ili_rate` has one value per ISO week, so a daily CCF would mostly show the repeated values within a week. Weekly arrivals = mean daily episodes in complete ISO weeks |
| Scale of skewed exposures | `pm10` and `pm25` on log, `ili_rate` and `flu_index` on log(1 + x); `temp_mean` as is. The raw scale is a sensitivity analysis | a Pearson CCF on a strongly skewed series is driven by a few extreme days or weeks (skewness: PM10 2.4, ILI 3.9, with many zero weeks; after the log about 0.1 and 0.7, see §1) |
| Period | each analysis uses the span where both series are observed (ILI from 2022-06) | seasonal adjustment of both series on the same days |
| Temperature adjustment for H3 | natural cubic spline of `temp_mean` (3 df) added to the seasonal design of both series; the unadjusted version is a sensitivity analysis | standard for air pollution (temperature drives both PM10 and arrivals); decided 2026-11-09 after a first run of this notebook showed PM10 correlating with injuries (deviations log, item 27) |
| Negative-control outcomes | injuries by unit (`arr_injury_mts_all`, KPP) and by diagnosis (`arr_dx_injury_icd`, ICD-10 S/T), for H2 and H3 | preregistered "injuries as a negative-control outcome"; both definitions are used because they differ (unit of the final visit vs diagnosis; diagnoses exclude Hitri pregledi) |
| Zero-count days | treated as missing (log scale); one day in INP (2021-05-08), none in the other streams | log(0) is undefined |
| Missing days | not filled: a filtered value is missing if any day it uses is missing | consistent with no interpolation in the primary variables (deviation 7) |

## Imports

In [ ]:
import hashlib
import json
import platform
import subprocess
from datetime import datetime, timezone
from importlib.metadata import version as pkg_version
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import patsy
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.seasonal import MSTL

## Configuration

In [ ]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements-lock.txt").exists())
PROCESSED = ROOT / "data" / "processed"
OUTPUTS = ROOT / "outputs" / "03_1_cross_correlation"
OUTPUTS.mkdir(parents=True, exist_ok=True)

PANEL_IN = PROCESSED / "daily_panel_lagged.parquet"
CODEBOOK_IN = PROCESSED / "daily_panel_lagged_codebook.csv"
RUN_INFO_02 = ROOT / "outputs" / "02_lag_variables" / "run_info.json"

# Removing seasonality (main specification and the DLNM-style sensitivity)
SEASON_SPECS = {
    "fourier": {"trend_df_per_year": 1, "fourier_harmonics": 3},   # main
    "spline7": {"trend_df_per_year": 7, "fourier_harmonics": 0},   # sensitivity: 7 df/year spline of time, no harmonics
}
EXPOSURE_TRANSFORM = {"pm10": "log", "pm25": "log", "ili_rate": "log1p", "flu_index": "log1p"}   # others as is
TRANSFORMS = {"log": np.log, "log1p": np.log1p}
TEMP_DF = 3                  # df of the temperature spline in the H3 adjustment
MAX_AR = {"D": 14, "W": 6}   # largest AR order considered for prewhitening
MAX_LAG = {"D": 21, "W": 3}  # lags shown: -MAX_LAG .. MAX_LAG (days or weeks)
N_SIM = 1000                 # dummy exposures per analysis
BREAKS = ["hp_referral_era", "triage_break_2024_06"]   # structural breaks (step indicators), part of the seasonal design
SEED = 20261109

# Analyses: (id, hypothesis, exposure, outcome, resolution D/W, role, variant)
# variant: season spec, or a subset of days
OUT_ALL, OUT_MED, OUT_INJ, OUT_INJ_ICD = "arr_all_mts_all", "arr_medical_mts_all", "arr_injury_mts_all", "arr_dx_injury_icd"
# H3 is adjusted for temperature (flag temp_adj): natural spline of temp_mean, TEMP_DF df, in the seasonal design
ANALYSES = pd.DataFrame([
    ("H1",               "H1", "temp_mean", OUT_ALL,     "D", "main",                     "fourier"),
    ("H2",               "H2", "ili_rate",  OUT_MED,     "W", "main",                     "fourier"),
    ("H3",               "H3", "pm10",      OUT_MED,     "D", "main",                     "fourier|temp_adj"),
    ("H2_injury_kpp",    "H2", "ili_rate",  OUT_INJ,     "W", "negative-control outcome", "fourier"),
    ("H2_injury_icd",    "H2", "ili_rate",  OUT_INJ_ICD, "W", "negative-control outcome", "fourier"),
    ("H3_injury_kpp",    "H3", "pm10",      OUT_INJ,     "D", "negative-control outcome", "fourier|temp_adj"),
    ("H3_injury_icd",    "H3", "pm10",      OUT_INJ_ICD, "D", "negative-control outcome", "fourier|temp_adj"),
    ("H1_spline7",       "H1", "temp_mean", OUT_ALL,     "D", "sensitivity",              "spline7"),
    ("H2_spline7",       "H2", "ili_rate",  OUT_MED,     "W", "sensitivity",              "spline7"),
    ("H3_spline7",       "H3", "pm10",      OUT_MED,     "D", "sensitivity",              "spline7|temp_adj"),
    ("H1_nocovid",       "H1", "temp_mean", OUT_ALL,     "D", "sensitivity",              "fourier|no_covid"),
    ("H3_nocovid",       "H3", "pm10",      OUT_MED,     "D", "sensitivity",              "fourier|no_covid|temp_adj"),
    ("H2_flu_index",     "H2", "flu_index", OUT_MED,     "W", "sensitivity",              "fourier"),
    ("H3_verified",      "H3", "pm10",      OUT_MED,     "D", "sensitivity",              "fourier|verified_pm|temp_adj"),
    ("H2_raw_scale",     "H2", "ili_rate",  OUT_MED,     "W", "sensitivity",              "fourier|raw_scale"),
    ("H3_raw_scale",     "H3", "pm10",      OUT_MED,     "D", "sensitivity",              "fourier|raw_scale|temp_adj"),
    ("H3_no_temp",       "H3", "pm10",      OUT_MED,     "D", "sensitivity",              "fourier"),
    ("H3_injury_kpp_no_temp", "H3", "pm10", OUT_INJ,     "D", "sensitivity",              "fourier"),
    # PM2.5: secondary exposure for H3, analysed exactly like PM10 (deviations log, item 39)
    ("PM25",             "PM2.5", "pm25",   OUT_MED,     "D", "secondary",                "fourier|temp_adj"),
    ("PM25_injury_kpp",  "PM2.5", "pm25",   OUT_INJ,     "D", "negative-control outcome", "fourier|temp_adj"),
    ("PM25_injury_icd",  "PM2.5", "pm25",   OUT_INJ_ICD, "D", "negative-control outcome", "fourier|temp_adj"),
    ("PM25_spline7",     "PM2.5", "pm25",   OUT_MED,     "D", "sensitivity",              "spline7|temp_adj"),
    ("PM25_nocovid",     "PM2.5", "pm25",   OUT_MED,     "D", "sensitivity",              "fourier|no_covid|temp_adj"),
    ("PM25_verified",    "PM2.5", "pm25",   OUT_MED,     "D", "sensitivity",              "fourier|verified_pm|temp_adj"),
    ("PM25_raw_scale",   "PM2.5", "pm25",   OUT_MED,     "D", "sensitivity",              "fourier|raw_scale|temp_adj"),
    ("PM25_no_temp",     "PM2.5", "pm25",   OUT_MED,     "D", "sensitivity",              "fourier"),
    ("PM25_injury_kpp_no_temp", "PM2.5", "pm25", OUT_INJ, "D", "sensitivity",             "fourier"),
], columns=["id", "hypothesis", "exposure", "outcome", "res", "role", "variant"]).set_index("id")

COLORS = {"blue": "#2a78d6", "orange": "#eb6834", "aqua": "#1baf7a", "grey": "#8a8984", "neutral": "#f0efec", "red": "#e34948"}
plt.rcParams.update({
    "figure.figsize": (12, 3.6), "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False, "font.size": 9,
})
display(ANALYSES)

## 1. Load and check

The run stops if the panel is not the file `02_lag_variables.ipynb` last wrote (SHA-256 against its `run_info.json`). Days in an HIS extraction gap (`his_gap`) already have missing outcomes.

In [ ]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for block in iter(lambda: fh.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


run_info_02 = json.loads(RUN_INFO_02.read_text())
panel_sha = sha256(PANEL_IN)
if panel_sha != run_info_02["panel_sha256"]:
    raise RuntimeError(f"{PANEL_IN.name} differs from the panel recorded in {RUN_INFO_02.relative_to(ROOT)} "
                       "(run 02_lag_variables.ipynb again).")

panel = pd.read_parquet(PANEL_IN)
codebook = pd.read_csv(CODEBOOK_IN).set_index("column")
DESIGN_COLS = ["time", "dow", "holiday", "covid", *BREAKS, "temp_mean"]
needed = set(ANALYSES["exposure"]) | set(ANALYSES["outcome"]) | set(DESIGN_COLS) | {"pm_unverified_instrument"}
missing = needed - set(panel.columns)
if missing:
    raise RuntimeError(f"Columns missing from the panel: {sorted(missing)}")

print(f"Panel: {len(panel)} days ({panel.index.min().date()} to {panel.index.max().date()}); checksum matches 02's run record")
display(pd.DataFrame({c: {"first": panel[c].first_valid_index().date(), "last": panel[c].last_valid_index().date(),
                          "missing days": int(panel[c].isna().sum())} for c in sorted(needed - {"time", "dow"} - set(BREAKS))}).T)

# Skewness of the exposures on the raw and the analysis scale (weekly exposures: one value per ISO week)
skew = {}
for c in sorted(set(ANALYSES["exposure"])):
    v = panel[c].resample("W-SUN").first() if (ANALYSES.loc[ANALYSES["exposure"] == c, "res"] == "W").any() else panel[c]
    t = EXPOSURE_TRANSFORM.get(c)
    skew[c] = {"skewness raw": round(v.skew(), 2), "analysis scale": t or "as is",
               "skewness analysis scale": round(TRANSFORMS[t](v).skew(), 2) if t else round(v.skew(), 2)}
display(pd.DataFrame(skew).T)

## 2. Primary outcomes: daily, weekly and yearly level

Daily episodes (thin), their ISO-week mean and the calendar-year mean. Charts show aggregated counts only.

In [ ]:
OUTCOMES = [OUT_ALL, OUT_MED, OUT_INJ]
fig, axes = plt.subplots(len(OUTCOMES), 1, figsize=(12, 7.5), sharex=True)
for ax, c in zip(axes, OUTCOMES):
    y = panel[c]
    ax.plot(y.index, y, lw=0.4, color=COLORS["grey"], alpha=0.6, label="daily")
    wk = y.resample("W-SUN").mean()
    ax.plot(wk.index, wk, lw=1.5, color=COLORS["blue"], label="weekly mean")
    yr = y.groupby(y.index.year).transform("mean")
    ax.plot(yr.index, yr, lw=2, color=COLORS["orange"], label="yearly mean")
    ax.set_title(f"{c}: episodes per day", loc="left")
axes[0].legend(ncol=3, loc="upper left")
fig.tight_layout()
fig.savefig(OUTPUTS / "outcomes_levels.png")
plt.show()

summary = pd.DataFrame({c: panel[c].groupby(panel.index.year).mean().round(1) for c in OUTCOMES}).rename_axis("year")
display(summary)

## 3. Decomposition: trend, weekly and yearly seasonality

MSTL (multiple seasonal-trend decomposition by LOESS; periods 7 and 365 days) of log daily episodes. MSTL needs a complete series, so the few missing outcome days are linearly interpolated **for this figure only**; no later step uses the interpolated values.

In [ ]:
fig, axes = plt.subplots(4, len(OUTCOMES), figsize=(13, 8), sharex=True)
for j, c in enumerate(OUTCOMES):
    y = np.log(panel[c]).interpolate(limit_area="inside")
    y = y.loc[y.first_valid_index():y.last_valid_index()]
    res = MSTL(y, periods=(7, 365)).fit()
    parts = {"trend": res.trend, "weekly": res.seasonal["seasonal_7"], "yearly": res.seasonal["seasonal_365"], "residual": res.resid}
    for i, (name, s) in enumerate(parts.items()):
        ax = axes[i, j]
        ax.plot(s.index, s, lw=0.6 if name in ("weekly", "residual") else 1.5, color=COLORS["blue"])
        if j == 0:
            ax.set_ylabel(name)
        if i == 0:
            ax.set_title(c, loc="left")
    var = {k: v.var() for k, v in parts.items()}
    print(f"{c}: share of variance of log episodes (components are not exactly orthogonal): "
          + ", ".join(f"{k} {v / y.var():.0%}" for k, v in var.items()))
fig.suptitle("MSTL decomposition of log daily episodes", x=0.01, ha="left")
fig.tight_layout()
fig.savefig(OUTPUTS / "outcomes_mstl.png")
plt.show()

## 4. Removing seasonality, and autocorrelation of the adjusted arrivals

Each series is regressed (OLS) on the seasonal design: an intercept, a natural cubic spline of time, annual Fourier harmonics, day of week, holidays, the COVID-19 period indicator and the structural-break indicators (weekly series: holiday days per week and the weekly share of the indicators, instead of day of week). Indicators that are constant over an analysis' span are left out. The residuals are the **seasonally adjusted** series. The same design is used for the exposure and the arrivals, so both are adjusted for the same confounders.

ACF and PACF of the adjusted arrivals show the autocorrelation left after seasonal adjustment. This is what the DLNM must handle (lagged residuals or an autoregressive term, Metodologija), and why a raw CCF would be misleading.

In [ ]:
YEARS = len(panel) / 365.25


def weekly(df: pd.DataFrame) -> pd.DataFrame:
    """ISO-week series from daily columns x, y and the design columns: y = mean daily episodes over complete weeks,
    x = the week's value; holidays summed, other indicators averaged."""
    g = df.groupby(df.index.to_period("W-SUN"))
    w = pd.DataFrame({
        "y": g["y"].mean().where(g["y"].count() == 7),
        "x": g["x"].first().where((g["x"].nunique() <= 1) & (g["x"].count() == 7)),
        "time": g["time"].mean(),
        "holiday": g["holiday"].sum(),
        **{c: g[c].mean() for c in ["covid", *BREAKS]},
        "n_days": g.size(),
    })
    w = w[w["n_days"] == 7].drop(columns="n_days")
    w.index = w.index.to_timestamp(how="start")
    return w


def season_design(df: pd.DataFrame, spec: str, res: str) -> np.ndarray:
    s = SEASON_SPECS[spec]
    df_trend = max(1, round(s["trend_df_per_year"] * YEARS))
    terms = [f"cr(time, df={df_trend})"] if df_trend >= 3 else ["time"]
    X = patsy.dmatrix(" + ".join(terms), df, return_type="dataframe")
    phase = 2 * np.pi * df["time"].to_numpy() / 365.25
    for k in range(1, s["fourier_harmonics"] + 1):
        X[f"sin{k}"], X[f"cos{k}"] = np.sin(k * phase), np.cos(k * phase)
    if res == "D":
        X = X.join(pd.get_dummies(df["dow"].astype(int), prefix="dow", drop_first=True, dtype=float))
    X["holiday"] = df["holiday"].to_numpy(dtype=float)
    for c in ["covid", *BREAKS]:
        if df[c].nunique() > 1:   # leave out indicators that are constant over the span
            X[c] = df[c].to_numpy(dtype=float)
    return X.to_numpy(dtype=float)


def adjust(v: np.ndarray, X: np.ndarray) -> np.ndarray:
    """Residuals of OLS of v on X (missing where v is missing)."""
    ok = ~np.isnan(v)
    beta, *_ = np.linalg.lstsq(X[ok], v[ok], rcond=None)
    r = np.full_like(v, np.nan)
    r[ok] = v[ok] - X[ok] @ beta
    return r


fig, axes = plt.subplots(len(OUTCOMES), 2, figsize=(12, 7.5))
for i, c in enumerate(OUTCOMES):
    X = season_design(panel, "fourier", "D")
    r = pd.Series(adjust(np.log(panel[c].to_numpy(dtype=float)), X), index=panel.index)
    plot_acf(r, lags=28, missing="conservative", ax=axes[i, 0], color=COLORS["blue"], vlines_kwargs={"colors": COLORS["blue"]})
    plot_pacf(r.dropna(), lags=28, ax=axes[i, 1], color=COLORS["blue"], vlines_kwargs={"colors": COLORS["blue"]}, method="ywm")
    axes[i, 0].set_title(f"ACF, adjusted log {c}", loc="left")
    axes[i, 1].set_title(f"PACF, adjusted log {c}", loc="left")
fig.tight_layout()
fig.savefig(OUTPUTS / "outcomes_acf_pacf.png")
plt.show()

## 5. Prewhitened cross-correlation

For each analysis:

1. Build the series (daily, or weekly for influenza), drop the days excluded by the variant, put skewed exposures on the log scale, keep the span where both series are observed, and adjust exposure *x* and log arrivals *y* for seasonality (§4).
2. Fit AR(*p*) to adjusted *x* by least squares, *p* = 0 … `MAX_AR` chosen by AIC on a common sample.
3. Filter both adjusted series with the same AR filter: x̃_t = x_t − c − Σ φ_j x_{t−j}, and likewise ỹ. x̃ is then close to white noise, so the CCF of x̃ and ỹ is not inflated by autocorrelation.
4. r(k) = corr(x̃_t, ỹ_{t+k}) for k = −L … L, on all complete pairs (n_k). Pointwise limits ±1.96/√n_k.
5. **Dummy exposure** (preregistered negative control): `N_SIM` series simulated from the fitted AR(*p*) model of *x*, with *x*'s missing days, are put through steps 2–4 with the real *y* (AR coefficients re-estimated each time). Their CCFs give pointwise 2.5–97.5% limits and a **simultaneous** 95% limit: the 95th percentile of max |r(k)| over k = 0 … L.

In [ ]:
def prepare(exposure: str, outcome: str, res: str, variant: str = "fourier", transform: str | None = "default") -> dict:
    """Seasonally adjusted exposure x and log outcome y on the span where both are observed.
    Variants blank out days of x and y only; the design columns are never changed."""
    spec, *flags = variant.split("|")
    x, y = panel[exposure].astype(float).copy(), panel[outcome].astype(float).copy()
    if "no_covid" in flags:
        x[panel["covid"] == 1] = np.nan
        y[panel["covid"] == 1] = np.nan
    if "verified_pm" in flags:
        x[panel["pm_unverified_instrument"] == 1] = np.nan
    t = EXPOSURE_TRANSFORM.get(exposure) if transform == "default" else transform
    if t and "raw_scale" not in flags:
        x = TRANSFORMS[t](x)
    df = panel[DESIGN_COLS].assign(x=x, y=y)
    if res == "W":
        df = weekly(df)
    both = df["x"].notna() & df["y"].notna()   # span where both series are observed
    first, last = both[both].index[[0, -1]]
    df = df.loc[first:last]
    X = season_design(df, spec, res)
    if "temp_adj" in flags:   # H3: temperature adjustment, natural spline of temp_mean
        X = np.column_stack([X, patsy.dmatrix(f"cr(t, df={TEMP_DF}) - 1", {"t": df["temp_mean"].to_numpy()}, return_type="matrix")])
    y_pos = df["y"].where(df["y"] > 0)   # a zero-count day (one INP day) is treated as missing: log(0) is undefined
    return {"x": adjust(df["x"].to_numpy(), X), "y": adjust(np.log(y_pos.to_numpy()), X),
            "period": f"{first.date()} to {last.date()}"}


def prewhitened_ccf(d: dict, res: str) -> dict:
    fit = fit_ar(d["x"], MAX_AR[res])
    xf, yf = ar_filter(d["x"], fit["c"], fit["phi"]), ar_filter(d["y"], 0.0, fit["phi"])
    lags = np.arange(-MAX_LAG[res], MAX_LAG[res] + 1)
    r, n = ccf(xf, yf, lags)
    return {"fit": fit, "xf": xf, "yf": yf, "lags": lags, "r": r, "n": n}


def lag_matrix(v: np.ndarray, p: int) -> np.ndarray:
    L = np.full((len(v), p), np.nan)
    for j in range(1, p + 1):
        L[j:, j - 1] = v[:-j]
    return L


def fit_ar(v: np.ndarray, max_p: int) -> dict:
    """AR(p) by least squares; p by AIC on the rows complete for max_p lags."""
    while True:   # lower max_p until enough rows have all lags (series with many short gaps)
        Lm = lag_matrix(v, max_p)
        common = ~np.isnan(v) & ~np.isnan(Lm).any(axis=1)
        if common.sum() >= 10 * (max_p + 1) or max_p == 0:
            break
        max_p -= 1
    aic = {}
    for p in range(max_p + 1):
        A = np.column_stack([np.ones(common.sum()), Lm[common, :p]])
        beta, *_ = np.linalg.lstsq(A, v[common], rcond=None)
        rss = np.sum((v[common] - A @ beta) ** 2)
        aic[p] = common.sum() * np.log(rss / common.sum()) + 2 * (p + 1)
    p = min(aic, key=aic.get)
    return {"p": p, **ar_coefs(v, p)}


def ar_coefs(v: np.ndarray, p: int) -> dict:
    Lm = lag_matrix(v, p)
    ok = ~np.isnan(v) & ~np.isnan(Lm).any(axis=1)
    A = np.column_stack([np.ones(ok.sum()), Lm[ok]])
    beta, *_ = np.linalg.lstsq(A, v[ok], rcond=None)
    return {"c": beta[0], "phi": beta[1:], "sigma": np.std(v[ok] - A @ beta, ddof=p + 1)}


def ar_filter(v: np.ndarray, c: float, phi: np.ndarray) -> np.ndarray:
    """v_t - c - sum phi_j v_{t-j}; missing if any value used is missing."""
    return v - c - (lag_matrix(v, len(phi)) @ phi if len(phi) else 0)


def ccf(a: np.ndarray, b: np.ndarray, lags: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """r(k) = corr(a_t, b_{t+k}) over complete pairs; returns r and n."""
    r, n = np.full(len(lags), np.nan), np.zeros(len(lags), dtype=int)
    for i, k in enumerate(lags):
        x, y = (a[:len(a) - k], b[k:]) if k >= 0 else (a[-k:], b[:len(b) + k])
        ok = ~np.isnan(x) & ~np.isnan(y)
        n[i] = ok.sum()
        if n[i] > 3:
            r[i] = np.corrcoef(x[ok], y[ok])[0, 1]
    return r, n


def simulate_ar(n: int, fit: dict, rng: np.random.Generator, burn: int = 500) -> np.ndarray:
    e = rng.normal(0, fit["sigma"], n + burn)
    v, p = np.zeros(n + burn), fit["p"]
    for t in range(n + burn):
        v[t] = e[t] + sum(fit["phi"][j] * v[t - j - 1] for j in range(p) if t - j - 1 >= 0)
    return v[burn:]


def run(aid: str, a: pd.Series, rng: np.random.Generator) -> dict:
    prep = prepare(a["exposure"], a["outcome"], a["res"], a["variant"])
    c = prewhitened_ccf(prep, a["res"])
    x, y, fit, xf, lags, r, n = prep["x"], prep["y"], c["fit"], c["xf"], c["lags"], c["r"], c["n"]

    # Dummy exposure: same AR model and missing days as x
    miss = np.isnan(x)
    sims = np.empty((N_SIM, len(lags)))
    for s in range(N_SIM):
        d = simulate_ar(len(x), fit, rng)
        d[miss] = np.nan
        f = ar_coefs(d, fit["p"])
        sims[s], _ = ccf(ar_filter(d, f["c"], f["phi"]), ar_filter(y, 0.0, f["phi"]), lags)
    pos = lags >= 0
    resid_acf1 = ccf(xf, xf, np.array([1]))[0][0]
    return {
        "id": aid, "lags": lags, "r": r, "n": n, "xf": xf, "yf": c["yf"],
        "pointwise": 1.96 / np.sqrt(n),
        "sim_lo": np.nanpercentile(sims, 2.5, axis=0), "sim_hi": np.nanpercentile(sims, 97.5, axis=0),
        "simultaneous": np.nanpercentile(np.nanmax(np.abs(sims[:, pos]), axis=1), 95),
        "ar_p": fit["p"], "prewhitened_x_acf1": resid_acf1,
        "period": prep["period"],
    }


rng = np.random.default_rng(SEED)
results = {aid: run(aid, a, rng) for aid, a in ANALYSES.iterrows()}
print(f"{len(results)} analyses done, {N_SIM} dummy exposures each")

### Results table

For each analysis: AR order, the lag-1 autocorrelation left in the prewhitened exposure (close to 0 = prewhitening worked), the strongest correlation at lags 0 … L with its 95% CI (Fisher *z*), the lags 0 … L outside the pointwise limits, whether any lag 0 … L crosses the simultaneous limit, and the same count for negative lags (arrivals first). Unit of *L*: days, or weeks for influenza.

In [ ]:
rows, long = [], []
for aid, res in results.items():
    a = ANALYSES.loc[aid]
    lags, r, n, lim = res["lags"], res["r"], res["n"], res["pointwise"]
    pos, neg = lags >= 0, lags < 0
    i = np.nanargmax(np.where(pos, np.abs(r), -np.inf))
    z, se = np.arctanh(r[i]), 1 / np.sqrt(n[i] - 3)
    outside = lags[pos & (np.abs(r) > lim)]
    rows.append({
        "id": aid, "hypothesis": a["hypothesis"], "role": a["role"], "exposure": a["exposure"], "outcome": a["outcome"],
        "unit": "week" if a["res"] == "W" else "day", "period": res["period"], "n": int(n[lags == 0][0]),
        "AR p": res["ar_p"], "prewhitened x ACF(1)": round(res["prewhitened_x_acf1"], 3),
        "peak lag": int(lags[i]), "peak r": round(r[i], 3),
        "peak 95% CI": f"{np.tanh(z - 1.96 * se):.3f} to {np.tanh(z + 1.96 * se):.3f}",
        "lags outside pointwise": ", ".join(map(str, outside)) or "none",
        "simultaneous limit": round(res["simultaneous"], 3),
        "crosses simultaneous": bool(np.nanmax(np.abs(r[pos])) > res["simultaneous"]),
        "negative lags outside pointwise": int((neg & (np.abs(r) > lim)).sum()),
    })
    long.append(pd.DataFrame({"id": aid, "lag": lags, "r": r, "n": n, "pointwise_limit": lim,
                              "dummy_lo": res["sim_lo"], "dummy_hi": res["sim_hi"], "simultaneous_limit": res["simultaneous"]}))

table = pd.DataFrame(rows).set_index("id")
ccf_long = pd.concat(long, ignore_index=True)
table.to_csv(OUTPUTS / "ccf_summary.csv")
ccf_long.round(5).to_csv(OUTPUTS / "ccf_values.csv", index=False)
with pd.option_context("display.max_columns", 30, "display.width", 250):
    display(table)

## 6. CCF plots

Bars: r(k). Grey dashed: pointwise ±1.96/√n. Grey band: pointwise 2.5–97.5% of the dummy exposures (should match the dashed line when prewhitening works). Orange dotted: simultaneous 95% limit for lags 0 … L. The shaded left half (negative lags, arrivals first) is the reverse-direction check.

In [ ]:
def ccf_plot(ax, res: dict, title: str, unit: str):
    lags, r = res["lags"], res["r"]
    ax.axvspan(lags[0] - 0.5, -0.5, color=COLORS["neutral"], zorder=0)
    ax.fill_between(lags, res["sim_lo"], res["sim_hi"], step="mid", color=COLORS["grey"], alpha=0.25, lw=0, label="dummy exposure 95%")
    ax.plot(lags, res["pointwise"], ls="--", lw=1, color=COLORS["grey"], label="±1.96/√n")
    ax.plot(lags, -res["pointwise"], ls="--", lw=1, color=COLORS["grey"])
    for s in (1, -1):
        ax.hlines(s * res["simultaneous"], 0, lags[-1], color=COLORS["orange"], ls=":", lw=1.5,
                  label="simultaneous 95% (lags ≥ 0)" if s == 1 else None)
    ax.bar(lags, r, width=0.6, color=COLORS["blue"], zorder=3)
    ax.axhline(0, color="black", lw=0.6)
    ax.set_title(title, loc="left")
    ax.set_xlabel(f"lag ({unit}s; exposure first →)")


main_ids = [i for i in ANALYSES.index if ANALYSES.loc[i, "role"] != "sensitivity"]
sens_ids = [i for i in ANALYSES.index if ANALYSES.loc[i, "role"] == "sensitivity"]
for name, ids in (("main_and_negative_control", main_ids), ("sensitivity", sens_ids)):
    ncol = 3
    nrow = int(np.ceil(len(ids) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(13, 3.2 * nrow), squeeze=False)
    for ax, aid in zip(axes.flat, ids):
        a = ANALYSES.loc[aid]
        ccf_plot(ax, results[aid], f"{aid}: {a['exposure']} → {a['outcome']}", "week" if a["res"] == "W" else "day")
    for ax in axes.flat[len(ids):]:
        ax.set_visible(False)
    axes.flat[0].legend(loc="upper right", fontsize=7)
    fig.suptitle(f"Prewhitened cross-correlation ({name.replace('_', ' ')})", x=0.01, ha="left")
    fig.tight_layout()
    fig.savefig(OUTPUTS / f"ccf_{name}.png")
    plt.show()

## 7. Collinearity of exposures (Spearman)

Spearman correlations between the daily primary and secondary weather, air-quality and respiratory exposures, on the raw values (top) and after removing seasonality (bottom). High raw correlations that disappear after adjustment are shared seasonality; those that remain (e.g. temperature with O₃) need care when both enter one model.

In [ ]:
EXPO_SOURCES = {"arso_weather", "era5", "arso_air", "erviss", "flu_index"}
base = codebook[(codebook["transform"] == "none") & codebook["source"].isin(EXPO_SOURCES)
                & codebook["use"].isin(["primary", "secondary"]) & (codebook["unit"] != "0/1")]
base = base[~base.index.str.startswith(("det_", "pos_"))]   # keep the rate (ili_rate, ari_rate) and the index; detections/positivity are near copies
expo = base.index.tolist()

Xd = season_design(panel, "fourier", "D")
adjusted = pd.DataFrame({c: adjust(panel[c].to_numpy(dtype=float), Xd) for c in expo}, index=panel.index)
corr_raw = panel[expo].corr(method="spearman")
corr_adj = adjusted.corr(method="spearman")
corr_raw.round(3).to_csv(OUTPUTS / "exposure_spearman_raw.csv")
corr_adj.round(3).to_csv(OUTPUTS / "exposure_spearman_adjusted.csv")

from matplotlib.colors import LinearSegmentedColormap
div = LinearSegmentedColormap.from_list("div", [COLORS["blue"], COLORS["neutral"], COLORS["red"]])
fig, axes = plt.subplots(2, 1, figsize=(10, 17))
for ax, (name, m) in zip(axes, (("raw values", corr_raw), ("after removing seasonality", corr_adj))):
    im = ax.imshow(m, cmap=div, vmin=-1, vmax=1)
    ax.set_xticks(range(len(expo)), expo, rotation=90, fontsize=7)
    ax.set_yticks(range(len(expo)), expo, fontsize=7)
    ax.grid(False)
    for i in range(len(expo)):
        for j in range(len(expo)):
            if i != j and abs(m.iat[i, j]) >= 0.5:
                ax.text(j, i, f"{m.iat[i, j]:.1f}", ha="center", va="center", fontsize=5.5)
    ax.set_title(f"Spearman correlation of exposures, {name} (|ρ| ≥ 0.5 labelled)", loc="left")
    fig.colorbar(im, ax=ax, shrink=0.6)
fig.tight_layout()
fig.savefig(OUTPUTS / "exposure_spearman.png")
plt.show()

pairs = (corr_adj.where(np.triu(np.ones(corr_adj.shape, dtype=bool), 1)).stack().rename("rho_adjusted").to_frame()
         .join(corr_raw.stack().rename("rho_raw")))
strong = pairs[pairs["rho_adjusted"].abs() >= 0.5].sort_values("rho_adjusted", key=np.abs, ascending=False)
primary_expo = ["temp_mean", "pm10", "ili_rate"]
print("Pairs with |rho| >= 0.5 after removing seasonality that involve a primary exposure:")
display(strong[strong.index.get_level_values(0).isin(primary_expo) | strong.index.get_level_values(1).isin(primary_expo)].round(2))

## 8. Exploratory: all exposures × arrival streams

**Exploratory, not preregistered.** The same prewhitened CCF (main specification, §5) for every exposure in the panel against six arrival streams. With about 70 exposures × 6 streams, roughly 5% of pairs will cross the pointwise limits by chance, so results are judged against the **simultaneous** limit only, and nothing here is a finding without replication in the DLNM.

**Exposures** (rules fixed before the run): every column with `use` primary, secondary or exploratory, except outcomes and calendar columns, with at least 365 observed days and at least 30 non-zero days (drops rare warnings and events). Left out: `moon_illumination`, `moon_age_days`, `moon_phase` (smooth deterministic cycles, which an AR filter cannot prewhiten; the day indicators `full_moon`, `new_moon`, `full_moon_window` stay) and `sbi_return` (trading days only, so no daily AR filter can be fitted; stock returns belong to module B). Weekly series (ERVISS, flu index) are analysed weekly. Non-negative, non-binary series with skewness > 1 go on log(1 + x) (PM10: log, as in §5).

**Outcomes:** all arrivals, medical stream, injuries (KPP), Enota za bolezni, INP, Hitri pregledi (fast track; the referral break of 2022-11-19 is in the seasonal design).

**Simultaneous limit** without simulation (one per pair would take hours): Bonferroni over lags 0 … L, z(1 − 0.025 / (L + 1)) / √n. The first cell below compares it with the simulated limits of §5. The same limit is applied to the negative lags (arrivals first) as a reverse-direction check.

**Caution in reading.** Some exposures are partly the outcome itself or its consequence: EMS calls bring patients to the ED, and hospital occupancy rises with ED admissions. Their correlations are not exposure effects.

In [ ]:
from scipy.stats import norm

def bonferroni_limit(n: np.ndarray, L: int) -> np.ndarray:
    return np.where(n > 3, norm.ppf(1 - 0.025 / (L + 1)) / np.sqrt(np.maximum(n, 1)), np.nan)

check = pd.DataFrame({aid: {"simulated": res["simultaneous"],
                            "Bonferroni": bonferroni_limit(res["n"][res["lags"] == 0], MAX_LAG[ANALYSES.loc[aid, "res"]])[0]}
                      for aid, res in results.items()}).T.round(3)
print("Simultaneous 95% limit, simulated (§5) vs Bonferroni:")
display(check.T)

In [ ]:
GRID_OUTCOMES = {"all": OUT_ALL, "medical": OUT_MED, "injury": OUT_INJ, "disease": "arr_disease_mts_all",
                 "inp": "arr_inp_mts_all", "fasttrack": "arr_fasttrack_mts_all"}
GRID_EXCLUDE = {"moon_illumination", "moon_age_days", "moon_phase",   # deterministic smooth cycles
                "sbi_return"}   # trading days only (no run longer than 5 days); analysed in module B
MIN_DAYS, MIN_NONZERO = 365, 30

base = codebook[(codebook["transform"] == "none") & ~codebook["source"].isin(["outcomes", "calendar"])
                & codebook["use"].isin(["primary", "secondary", "exploratory"])]
iso_week = panel.index.to_period("W-SUN")
grid_expo = []
for c in base.index:
    s = panel[c]
    if c in GRID_EXCLUDE or s.notna().sum() < MIN_DAYS or (s.fillna(0) != 0).sum() < MIN_NONZERO:
        continue
    res = "W" if s.groupby(iso_week).nunique().max() <= 1 else "D"
    binary = set(s.dropna().unique()) <= {0, 1}
    if c in EXPOSURE_TRANSFORM:
        t = EXPOSURE_TRANSFORM[c]
    else:
        v = s.groupby(iso_week).first() if res == "W" else s
        t = "log1p" if (not binary and s.min() >= 0 and v.skew() > 1) else None
    grid_expo.append({"exposure": c, "source": base.loc[c, "source"], "use": base.loc[c, "use"], "res": res, "transform": t or "as is"})
grid_expo = pd.DataFrame(grid_expo).set_index("exposure")
print(f"{len(grid_expo)} exposures ({(grid_expo['res'] == 'W').sum()} weekly) x {len(GRID_OUTCOMES)} outcomes")
display(grid_expo.groupby(["source", "res"]).size().rename("exposures").to_frame().T)

rows, prewhitened = [], {}
for c, e in grid_expo.iterrows():
    L = MAX_LAG[e["res"]]
    for oname, o in GRID_OUTCOMES.items():
        d = prepare(c, o, e["res"], "fourier", None if e["transform"] == "as is" else e["transform"])
        cc = prewhitened_ccf(d, e["res"])
        lags, r, n = cc["lags"], cc["r"], cc["n"]
        lim = bonferroni_limit(n, L)
        pos, neg = lags >= 0, lags < 0
        i = np.nanargmax(np.where(pos, np.abs(r), -np.inf))
        j = np.nanargmax(np.where(neg, np.abs(r), -np.inf))
        p_lag = 2 * norm.sf(np.abs(r) * np.sqrt(n))   # per lag, white-noise null
        sidak = lambda p: -np.expm1(len(p) * np.log1p(-np.nanmin(p)))   # Šidák 1-(1-p_min)^k, exact for tiny p
        prewhitened[(c, oname)] = (cc["xf"], cc["yf"])
        rows.append({"p": sidak(p_lag[pos]), "p reverse": sidak(p_lag[neg]),"exposure": c, "outcome": oname, "source": e["source"], "use": e["use"],
                     "unit": "week" if e["res"] == "W" else "day", "transform": e["transform"], "period": d["period"],
                     "n": int(n[lags == 0][0]), "AR p": cc["fit"]["p"], "peak lag": int(lags[i]), "peak r": r[i],
                     "simultaneous limit": lim[i], "crosses": bool(np.abs(r[i]) > lim[i]),
                     "lags crossing": ", ".join(map(str, lags[pos & (np.abs(r) > lim)])),
                     "reverse peak lag": int(lags[j]), "reverse peak r": r[j], "reverse crosses": bool(np.abs(r[j]) > lim[j])})
grid = pd.DataFrame(rows)
grid.round(4).to_csv(OUTPUTS / "ccf_exploratory_grid.csv", index=False)
print(f"{len(grid)} pairs; {grid['crosses'].sum()} cross the simultaneous limit at lags >= 0, "
      f"{grid['reverse crosses'].sum()} at negative lags (arrivals first)")

### Heatmap

Colour: the strongest correlation at lags 0 … L (sign kept). A number marks a pair that crosses the simultaneous limit and gives its lag (days, or weeks for weekly exposures, marked w); `◂` marks a pair whose negative lags also cross it (reverse direction, a sign of confounding). Exposures are grouped by source.

In [ ]:
order_src = list(dict.fromkeys(grid_expo["source"]))
expo_order = grid_expo.assign(o=grid_expo["source"].map(order_src.index)).sort_values("o").index.tolist()
R = grid.pivot(index="exposure", columns="outcome", values="peak r").loc[expo_order, list(GRID_OUTCOMES)]
C = grid.pivot(index="exposure", columns="outcome", values="crosses").loc[expo_order, list(GRID_OUTCOMES)]
LAG = grid.pivot(index="exposure", columns="outcome", values="peak lag").loc[expo_order, list(GRID_OUTCOMES)]
REV = grid.pivot(index="exposure", columns="outcome", values="reverse crosses").loc[expo_order, list(GRID_OUTCOMES)]

vmax = max(0.15, float(np.nanmax(np.abs(R.to_numpy()))))
fig, ax = plt.subplots(figsize=(7.5, 0.21 * len(R) + 1.5))
im = ax.imshow(R, cmap=div, vmin=-vmax, vmax=vmax, aspect="auto")
ax.set_xticks(range(R.shape[1]), R.columns)
ax.xaxis.tick_top()
ax.set_yticks(range(len(R)), [f"{c} ({grid_expo.loc[c, 'source']})" for c in R.index], fontsize=7)
ax.grid(False)
for i, c in enumerate(R.index):
    w = "w" if grid_expo.loc[c, "res"] == "W" else ""
    for j, o in enumerate(R.columns):
        label = (f"{int(LAG.iat[i, j])}{w}" if C.iat[i, j] else "") + ("◂" if REV.iat[i, j] else "")
        if label:
            ax.text(j, i, label, ha="center", va="center", fontsize=6.5, fontweight="bold" if C.iat[i, j] else "normal")
bounds = np.flatnonzero(grid_expo.loc[R.index, "source"].ne(grid_expo.loc[R.index, "source"].shift()).to_numpy())[1:]
for b in bounds:
    ax.axhline(b - 0.5, color="white", lw=1.5)
fig.colorbar(im, ax=ax, shrink=0.3, label="peak r (lags ≥ 0)")
ax.set_title("Exploratory CCF screen\n(number = lag crossing the simultaneous limit)", loc="left", fontsize=9, pad=24)
fig.tight_layout()
fig.savefig(OUTPUTS / "ccf_exploratory_heatmap.png")
plt.show()

### Pairs crossing the simultaneous limit

Sorted by |r|. `reverse crosses` = the negative lags (arrivals first) also cross, so the pair more likely reflects shared timing or confounding than an effect of the exposure.

In [ ]:
hits = grid[grid["crosses"]].assign(abs_r=lambda d: d["peak r"].abs()).sort_values("abs_r", ascending=False)
with pd.option_context("display.max_rows", 200, "display.width", 250):
    display(hits[["exposure", "outcome", "source", "unit", "transform", "n", "peak lag", "peak r", "simultaneous limit",
                  "lags crossing", "reverse crosses"]].round(3).reset_index(drop=True))
print(f"Expected by chance at most about {0.05 * len(grid):.0f} of {len(grid)} pairs (5% per pair, simultaneous over lags).")

### Distribution checks: QQ plots

**QQ plot of the screen's p-values.** Each pair's p-value is the smallest two-sided p over lags 0 … L (white-noise null, r·√n ~ N(0, 1)), Šidák-corrected for the number of lags: 1 − (1 − p_min)^(L+1). After prewhitening the lags are close to independent, so this p is close to uniform when there is no association. Blue: lags 0 … L (exposure first). Grey: the same for the negative lags (arrivals first), which no exposure effect can produce, so they act as a **built-in null reference**: if they follow the diagonal, the test is calibrated, and blue points above them are more associations than chance. Grey band: 95% band for uniform order statistics. The 480 pairs are not independent (exposures correlate with each other and the streams overlap), so the band is a guide, not a test.

In [ ]:
from scipy.stats import beta as beta_dist

m = len(grid)
expected = (np.arange(1, m + 1) - 0.5) / m
ranks = np.arange(1, m + 1)
band_lo = beta_dist.ppf(0.025, ranks, m - ranks + 1)
band_hi = beta_dist.ppf(0.975, ranks, m - ranks + 1)
pv = np.sort(grid["p"].clip(lower=1e-300).to_numpy())
pv_rev = np.sort(grid["p reverse"].clip(lower=1e-300).to_numpy())

fig, ax = plt.subplots(figsize=(5.5, 5))
ax.fill_between(-np.log10(expected), -np.log10(band_hi), -np.log10(band_lo), color=COLORS["grey"], alpha=0.25, lw=0, label="95% band, no association")
ax.plot([0, -np.log10(expected[0])], [0, -np.log10(expected[0])], color=COLORS["grey"], lw=1)
ax.scatter(-np.log10(expected), -np.log10(pv_rev), s=10, color=COLORS["grey"], label="negative lags (arrivals first): null reference")
ax.scatter(-np.log10(expected), -np.log10(pv), s=12, color=COLORS["blue"], label="lags ≥ 0 (exposure first)")
ax.set_xlabel("expected −log10 p")
ax.set_ylabel("observed −log10 p")
ax.set_title(f"QQ plot of p-values, {m} exposure × outcome pairs", loc="left")
ax.legend(loc="upper left", fontsize=7)
fig.tight_layout()
fig.savefig(OUTPUTS / "ccf_exploratory_pvalue_qq.png")
plt.show()
print(f"p < 0.05: {(pv < 0.05).sum()} of {m} pairs at lags >= 0, {(pv_rev < 0.05).sum()} at negative lags (expected about {0.05 * m:.0f})")
print(f"Median p: {np.median(pv):.2f} at lags >= 0, {np.median(pv_rev):.2f} at negative lags (expected 0.50)")

**Normal QQ plots of the prewhitened series, for every pair that crosses the simultaneous limit** (main analyses of §5 first, then the exploratory screen sorted by |r|). Each panel shows the standardised prewhitened exposure x̃_t (blue) and filtered arrivals ỹ_{t+k} (orange) at the peak lag k, on the days used for r. The ±1.96/√n limits assume roughly normal, independent values; points far off the line in the tails mean a few extreme days carry much of the correlation. Binary or zero-heavy exposures (warnings, storms, pollen) cannot look normal; for them the **rank (Spearman) correlation at the same lag** (ρ, in the panel title and the table) is the check: if ρ is close to r, the association does not rest on a few extreme days.

In [ ]:
from scipy.stats import spearmanr


def at_lag(xf: np.ndarray, yf: np.ndarray, k: int) -> tuple[np.ndarray, np.ndarray]:
    x, y = (xf[:len(xf) - k], yf[k:]) if k >= 0 else (xf[-k:], yf[:len(yf) + k])
    ok = ~np.isnan(x) & ~np.isnan(y)
    return x[ok], y[ok]


def std(v: np.ndarray) -> np.ndarray:
    return (v - v.mean()) / v.std()


# Pairs: main analyses that cross, then exploratory hits by |r|
qq_items = []
for aid in table.index[table["crosses simultaneous"]]:
    res = results[aid]
    a = ANALYSES.loc[aid]
    qq_items.append({"label": f"{aid}: {a['exposure']} → {a['outcome'].replace('arr_', '').replace('_mts_all', '')}",
                     "xf": res["xf"], "yf": res["yf"], "k": int(table.loc[aid, "peak lag"]), "r": table.loc[aid, "peak r"],
                     "unit": "w" if a["res"] == "W" else "d", "set": "main", "exposure": a["exposure"], "outcome": a["outcome"]})
for _, h in hits.iterrows():
    xf, yf = prewhitened[(h["exposure"], h["outcome"])]
    qq_items.append({"label": f"{h['exposure']} → {h['outcome']}", "xf": xf, "yf": yf, "k": int(h["peak lag"]),
                     "r": h["peak r"], "unit": "w" if h["unit"] == "week" else "d", "set": "exploratory",
                     "exposure": h["exposure"], "outcome": h["outcome"]})

robust = []
for it in qq_items:
    x, y = at_lag(it["xf"], it["yf"], it["k"])
    it["x"], it["y"] = std(x), std(y)
    it["rho"] = spearmanr(x, y).statistic
    robust.append({"set": it["set"], "pair": it["label"], "lag": f"{it['k']}{it['unit']}", "n": len(x), "r": it["r"],
                   "Spearman rho": it["rho"], "excess kurtosis x": pd.Series(x).kurt(), "excess kurtosis y": pd.Series(y).kurt()})
robust = pd.DataFrame(robust)
robust["rho / r"] = robust["Spearman rho"] / robust["r"]
robust.round(3).to_csv(OUTPUTS / "ccf_hits_robustness.csv", index=False)

for name in ("main", "exploratory"):
    items = [it for it in qq_items if it["set"] == name]
    ncol = 6
    nrow = int(np.ceil(len(items) / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(14, 2.3 * nrow), squeeze=False)
    for ax, it in zip(axes.flat, items):
        q = norm.ppf((np.arange(1, len(it["x"]) + 1) - 0.5) / len(it["x"]))
        ax.plot([-4, 4], [-4, 4], color=COLORS["grey"], lw=0.8)
        ax.scatter(q, np.sort(it["x"]), s=3, color=COLORS["blue"], label="exposure x̃")
        ax.scatter(q, np.sort(it["y"]), s=3, color=COLORS["orange"], label="arrivals ỹ")
        ax.set_title(f"{it['label']}\nlag {it['k']}{it['unit']}, r={it['r']:.2f}, ρ={it['rho']:.2f}", fontsize=6.5, loc="left")
        ax.tick_params(labelsize=6)
        lim = max(4, np.ceil(max(np.abs(it["x"]).max(), np.abs(it["y"]).max())))
        ax.set_ylim(-lim, lim)
    for ax in axes.flat[len(items):]:
        ax.set_visible(False)
    axes.flat[0].legend(fontsize=6, loc="upper left", markerscale=3)
    fig.supxlabel("theoretical normal quantile", fontsize=8)
    fig.supylabel("standardised prewhitened value", fontsize=8)
    fig.suptitle(f"Normal QQ plots, {name} pairs crossing the simultaneous limit ({len(items)})", x=0.01, ha="left")
    fig.tight_layout(rect=(0, 0, 1, 1 - 0.45 / fig.get_figheight()))
    fig.savefig(OUTPUTS / f"ccf_qq_{name}.png")
    plt.show()

print("Pairs where the rank correlation is less than half of r (association may rest on extreme days):")
display(robust[robust["rho / r"] < 0.5].round(3))

### PCA of the exposure × outcome correlations

Which arrival streams respond alike, and which exposures drive the differences? Principal component analysis of the screen's correlation matrix: rows = exposures, columns = the six arrival streams, values = the strongest prewhitened correlation at lags 0 … L (sign kept, as in the heatmap). Each stream's column is centred but not scaled (all values are correlations on the same scale). Left out: `ems_calls`, `ems_vuz` and `hosp_occupancy`, which are part of, or a consequence of, the arrivals and would dominate the first component.

**Reading.** Arrows (loadings): streams pointing the same way have similar correlation profiles across exposures. Points (scores): exposures far out along an arrow correlate most with that stream relative to the others. The streams overlap (all arrivals include every unit; medical = disease + INP), so a large first component is expected. This is descriptive only.

In [ ]:
PCA_EXCLUDE = {"ems_calls", "ems_vuz", "hosp_occupancy"}
GROUP = {"arso_weather": "weather", "era5": "weather", "arso_warnings": "warnings", "meteoalarm": "warnings",
         "arso_air": "air quality", "erviss": "respiratory infection", "flu_index": "respiratory infection",
         "pcr": "respiratory infection", "covid": "respiratory infection", "pollen": "pollen",
         "ndvi": "other", "football": "other", "moon": "other"}
GROUP_COLORS = {"weather": COLORS["blue"], "warnings": "#eda100", "air quality": COLORS["aqua"],
                "respiratory infection": COLORS["orange"], "pollen": "#6250d6", "other": COLORS["grey"]}

M = R.drop(index=[c for c in R.index if c in PCA_EXCLUDE])
Mc = M - M.mean()
U, S, Vt = np.linalg.svd(Mc.to_numpy(), full_matrices=False)
explained = S ** 2 / np.sum(S ** 2)
loadings = pd.DataFrame(Vt.T, index=M.columns, columns=[f"PC{i + 1}" for i in range(len(S))])
scores = pd.DataFrame(U * S, index=M.index, columns=loadings.columns)
# Orient each component so that the all-arrivals loading is positive (sign is arbitrary in PCA)
for pc in loadings:
    if loadings.loc["all", pc] < 0:
        loadings[pc] *= -1
        scores[pc] *= -1
group = grid_expo.loc[M.index, "source"].map(GROUP)

ve = pd.DataFrame({"variance explained": explained, "cumulative": explained.cumsum()}, index=loadings.columns)
pd.concat([loadings.T, ve], axis=1).round(3).to_csv(OUTPUTS / "ccf_pca_loadings.csv")
scores.round(4).assign(group=group).to_csv(OUTPUTS / "ccf_pca_scores.csv")
print(f"{len(M)} exposures x {M.shape[1]} streams")
display(ve.round(3).T)
display(loadings[["PC1", "PC2", "PC3"]].round(3))

In [ ]:
fig, (ax0, ax) = plt.subplots(1, 2, figsize=(12, 7), gridspec_kw={"width_ratios": [1, 2]})

ax0.bar(range(1, len(explained) + 1), explained * 100, color=COLORS["blue"], width=0.6)
for i, v in enumerate(explained):
    ax0.text(i + 1, v * 100 + 1, f"{v:.0%}", ha="center", fontsize=7)
ax0.set_xticks(range(1, len(explained) + 1), loadings.columns)
ax0.set_ylabel("% of variance")
ax0.set_title("Variance explained", loc="left")

for g, col in GROUP_COLORS.items():
    sel = group == g
    if sel.any():
        ax.scatter(scores.loc[sel, "PC1"], scores.loc[sel, "PC2"], s=22, color=col, label=g, edgecolor="white", lw=0.6, zorder=3)
# label the exposures furthest from the origin
dist = np.hypot(scores["PC1"], scores["PC2"])
for c in dist.sort_values(ascending=False).index[:16]:
    ax.annotate(c, (scores.loc[c, "PC1"], scores.loc[c, "PC2"]), fontsize=6.5, xytext=(4, 3), textcoords="offset points")
scale = 0.85 * np.abs(scores[["PC1", "PC2"]]).to_numpy().max() / np.abs(loadings[["PC1", "PC2"]]).to_numpy().max()
arrows = loadings[["PC1", "PC2"]] * scale
labels = arrows * 1.1
# keep arrow labels apart (nudge vertically when two are closer than 4% of the plot height)
span = max(np.abs(scores[["PC1", "PC2"]]).to_numpy().max(), np.abs(arrows).to_numpy().max()) * 1.2
order = labels.sort_values("PC2").index
for prev, cur in zip(order[:-1], order[1:]):
    if abs(labels.loc[cur, "PC1"] - labels.loc[prev, "PC1"]) < 0.15 * span and labels.loc[cur, "PC2"] - labels.loc[prev, "PC2"] < 0.04 * 2 * span:
        labels.loc[cur, "PC2"] = labels.loc[prev, "PC2"] + 0.04 * 2 * span
for o in arrows.index:
    ax.annotate("", xy=tuple(arrows.loc[o]), xytext=(0, 0), arrowprops=dict(arrowstyle="->", color="black", lw=1.2))
    ax.text(labels.loc[o, "PC1"], labels.loc[o, "PC2"], o, fontsize=8.5, fontweight="bold", ha="center", va="center")
ax.set_xlim(-span, span)
ax.set_ylim(-span, span)
ax.set_aspect("equal")
ax.axhline(0, color=COLORS["grey"], lw=0.6)
ax.axvline(0, color=COLORS["grey"], lw=0.6)
ax.set_xlabel(f"PC1 ({explained[0]:.0%})")
ax.set_ylabel(f"PC2 ({explained[1]:.0%})")
ax.set_title("Biplot: exposures (points) and arrival streams (arrows)", loc="left")
ax.legend(fontsize=7, loc="best")
fig.tight_layout()
fig.savefig(OUTPUTS / "ccf_pca_biplot.png")
plt.show()

print("Exposures with the largest scores on PC1 and PC2:")
for pc in ("PC1", "PC2"):
    top = scores[pc].sort_values()
    print(f"  {pc} low:  " + ", ".join(f"{c} ({v:.2f})" for c, v in top.head(5).items()))
    print(f"  {pc} high: " + ", ".join(f"{c} ({v:.2f})" for c, v in top.tail(5)[::-1].items()))

## 9. Input for the DLNM lag choice

For each hypothesis, the main analysis next to its sensitivity analyses: the lags 0 … L outside the pointwise limits, and whether the simultaneous limit is crossed. These inform, but do not fix, the maximum lag of the DLNM; the preregistered temperature lag of about 21 days stays the default unless the DLNM's own checks (Q-AIC, residuals) say otherwise. Single lags outside the pointwise limits, with no neighbours and no crossing of the simultaneous limit, are expected by chance.

In [ ]:
display(table[["hypothesis", "role", "exposure", "outcome", "unit", "peak lag", "peak r", "lags outside pointwise",
               "crosses simultaneous", "negative lags outside pointwise"]].sort_values(["hypothesis", "role"]))

## 10. Run record

In [ ]:
def git(*args) -> str:
    try:
        return subprocess.run(["git", *args], cwd=ROOT, capture_output=True, text=True, check=True).stdout.strip()
    except Exception:  # noqa: BLE001
        return "unavailable"


PACKAGES = ["pandas", "numpy", "statsmodels", "patsy", "matplotlib"]
run_info = {
    "run_at": datetime.now(timezone.utc).astimezone().isoformat(timespec="seconds"),
    "git_commit": git("rev-parse", "HEAD"),
    "git_uncommitted_changes": bool(git("status", "--porcelain")),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "packages": {p: pkg_version(p) for p in PACKAGES},
    "settings": {"season_specs": SEASON_SPECS, "max_ar": MAX_AR, "max_lag": MAX_LAG, "n_sim": N_SIM, "seed": SEED,
                 "analyses": ANALYSES.reset_index().to_dict(orient="records")},
    "input_panel_sha256": panel_sha,
    "input_run_02": {"run_at": run_info_02["run_at"], "git_commit": run_info_02["git_commit"]},
    "outputs": {p.name: sha256(p) for p in sorted(OUTPUTS.glob("*.csv"))},
}
(OUTPUTS / "run_info.json").write_text(json.dumps(run_info, indent=2, ensure_ascii=False))
print(f"Saved results and {(OUTPUTS / 'run_info.json').relative_to(ROOT)}")